In [1]:
%load_ext dotenv
%dotenv

In [8]:
import langchain
import groq
import os
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableParallel

In [21]:
chat_template_books = ChatPromptTemplate.from_template('''
Suggest three of the best intermediate-level {programming language} books.
Answer only by listing the books.
''')

chat_template_projects = ChatPromptTemplate.from_template('''
Suggest three interesting {programming language} projects suitable for intermediate level programmers.
Answer only by listing the projects.
''')

chat_template_time = ChatPromptTemplate.from_template('''
I'am an intermediate level programmer.

Consider the following literature:
{books}

Also, consider the following projects:
{projects}

Roughly how much time would it take me to complete the literature and the projects?
''')

In [22]:
chat = ChatGroq(
    model_name="llama-3.1-8b-instant",  # Groq model
    temperature=0,
    max_tokens=500,
    model_kwargs={'seed':365},
    groq_api_key=os.getenv("GROQ_API_KEY"),  # remove if already set globally
)

In [23]:
string_parser = StrOutputParser()

In [24]:
chain_books = chat_template_books | chat | string_parser

chain_projects = chat_template_projects | chat | string_parser

In [25]:
chain_parallel = RunnableParallel({'books' : chain_books, 'projects': chain_projects})

In [26]:
chain_parallel.invoke({'programming language' : 'Python'})

{'books': '1. "Automate the Boring Stuff with Python" by Al Sweigart\n2. "Python Crash Course" by Eric Matthes\n3. "Learning Python" by Mark Lutz',
 'projects': '1. Web Scraper with Database Integration\n2. Chatbot using Natural Language Processing (NLP)\n3. Game Development with Pygame'}

In [31]:
chain_time1 = (RunnableParallel({'books' : chain_books, 'projects': chain_projects})
             | chat_template_time
             | chat
             | string_parser
             )

chain_time2 = ({'books' : chain_books, 'projects': chain_projects}
             | chat_template_time
             | chat
             | string_parser
             )

In [32]:
print(chain_time2.invoke({'programming language' : 'Python'}))

To estimate the time required to complete the literature and projects, let's break down the tasks into smaller components.

**Literature:**

1. "Automate the Boring Stuff with Python" by Al Sweigart:
	* Estimated reading time: 10-15 hours (depending on your reading speed and pace)
	* This book focuses on practical applications of Python, so you'll likely spend more time practicing and experimenting with the code.
2. "Python Crash Course" by Eric Matthes:
	* Estimated reading time: 20-25 hours (similar to the previous book, with a focus on practical applications)
	* This book covers a wide range of topics, so you'll need to spend more time reviewing and practicing the concepts.
3. "Learning Python" by Mark Lutz:
	* Estimated reading time: 30-40 hours (this book is a comprehensive resource, covering the language's syntax, semantics, and best practices)
	* As a comprehensive resource, you'll need to spend more time reviewing and practicing the concepts.

Assuming you'll spend an average o

In [33]:
chain_time2.get_graph().print_ascii()

            +-------------------------------+            
            | Parallel<books,projects>Input |            
            +-------------------------------+            
                   **               **                   
                ***                   ***                
              **                         **              
+--------------------+            +--------------------+ 
| ChatPromptTemplate |            | ChatPromptTemplate | 
+--------------------+            +--------------------+ 
           *                                 *           
           *                                 *           
           *                                 *           
     +----------+                      +----------+      
     | ChatGroq |                      | ChatGroq |      
     +----------+                      +----------+      
           *                                 *           
           *                                 *           
           *  

In [11]:
chain_parallel.get_graph().print_ascii()

            +-------------------------------+            
            | Parallel<books,projects>Input |            
            +-------------------------------+            
                   **               **                   
                ***                   ***                
              **                         **              
+--------------------+            +--------------------+ 
| ChatPromptTemplate |            | ChatPromptTemplate | 
+--------------------+            +--------------------+ 
           *                                 *           
           *                                 *           
           *                                 *           
     +----------+                      +----------+      
     | ChatGroq |                      | ChatGroq |      
     +----------+                      +----------+      
           *                                 *           
           *                                 *           
           *  